# 06 — Dataset 4 Extension: All-Pairs Cohort Setup and Parallel Experiments (17–26)

**Module Role (Extension):** Isolated sensitivity extension using all PTID-matched Storyteller × ECOG pairs (`dataset4_multimodal_cn_vs_noncn.csv`), running parallel versions of Experiments 17–26 with participant-grouped CV and writing outputs to separate `dataset4/` subfolders.

**Run Order:** Independent (run any time after `01_preprocessing.ipynb`).


## Setup: Load shared feature lists and configure output folders


In [15]:
# Four-dataset analysis script
# Retains the original cohort/model logic and code for Experiments 17-26.
# Earlier exploratory sections (Experiments 1-16), duplicate audit material,
# and the separate correlation/agreement objective (Experiment 27) are omitted.
# Dataset 4 is loaded from its separately prepared processed CSV.
# Dataset 4 experiment counterparts run in an isolated section below.

from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Locate the new project's processed data folder.
PROJECT_DIR = None
DATA_DIR = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (folder / "data" / "processed", folder / "processed_data"):
        if candidate.is_dir() and all(
            (candidate / filename).is_file()
            for filename in (
                "dataset1_storyteller.csv",
                "dataset2_ecog.csv",
                "primary_multimodal_30day_cohort.csv",
            )
        ):
            PROJECT_DIR = folder
            DATA_DIR = candidate
            break
    if DATA_DIR is not None:
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find a processed data folder containing the three dataset CSVs."
    )

RESULTS_ROOT = PROJECT_DIR / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Load the three datasets used by the retained analyses.
dataset1_storyteller = pd.read_csv(
    DATA_DIR / "dataset1_storyteller.csv", low_memory=False
)
dataset2_ecog = pd.read_csv(
    DATA_DIR / "dataset2_ecog.csv", low_memory=False
)
dataset3_multimodal = pd.read_csv(
    DATA_DIR / "primary_multimodal_30day_cohort.csv", low_memory=False
)

# Normalize identifiers and targets without changing cohort membership.
for df in (dataset1_storyteller, dataset2_ecog, dataset3_multimodal):
    if "PTID" not in df or "TARGET" not in df:
        raise KeyError("Each dataset must contain PTID and TARGET columns.")
    df["PTID"] = df["PTID"].astype("string").str.strip()
    df["TARGET"] = pd.to_numeric(df["TARGET"], errors="coerce")
    if df[["PTID", "TARGET"]].isna().any().any():
        raise ValueError("A dataset contains missing PTID or TARGET values.")
    if not df["TARGET"].isin([0, 1]).all():
        raise ValueError("TARGET must contain only 0 and 1.")
    df["TARGET"] = df["TARGET"].astype(int)
    for col in df.columns:
        if "DATE" in str(col).upper():
            df[col] = pd.to_datetime(df[col], errors="coerce")

# Apply the feature exclusions used in the original predictor audit.
STORY_EXCLUDED_TERMS = ("PLAYBACK_COUNT", "GMATCH", "LANGUAGE")
story_feature_cols = [
    col for col in dataset1_storyteller.columns
    if str(col).upper().startswith("STORY")
    and not any(term in str(col).upper() for term in STORY_EXCLUDED_TERMS)
    and col in dataset3_multimodal.columns
]
ecog_feature_cols = [f"ECOGPT{i}" for i in range(1, 13)]

for name, df, features in (
    ("Dataset 1", dataset1_storyteller, story_feature_cols),
    ("Dataset 2", dataset2_ecog, ecog_feature_cols),
    ("Dataset 3", dataset3_multimodal, story_feature_cols + ecog_feature_cols),
):
    missing = [col for col in features if col not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing predictor columns: {missing}")
    for col in features:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Loaded three datasets from:", DATA_DIR)
for name, df in (
    ("Dataset 1 - Storyteller", dataset1_storyteller),
    ("Dataset 2 - ECOG", dataset2_ecog),
    ("Dataset 3 - multimodal", dataset3_multimodal),
):
    print(
        f"{name}: {len(df)} rows, {df['PTID'].nunique()} participants, "
        f"targets={df['TARGET'].value_counts().sort_index().to_dict()}"
    )
print(
    f"Features: {len(story_feature_cols)} Storyteller, "
    f"{len(ecog_feature_cols)} ECOG, "
    f"{len(story_feature_cols) + len(ecog_feature_cols)} fusion"
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


Loaded three datasets from: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed
Dataset 1 - Storyteller: 1430 rows, 490 participants, targets={0: 1073, 1: 357}
Dataset 2 - ECOG: 501 rows, 397 participants, targets={0: 273, 1: 228}
Dataset 3 - multimodal: 77 rows, 71 participants, targets={0: 53, 1: 24}
Features: 51 Storyteller, 12 ECOG, 63 fusion


C:\Users\amnar\AppData\Local\Temp\ipykernel_86684\698587727.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


## Dataset 4: all-pairs cohort setup

Loads and validates the prepared CN-versus-non-CN Storyteller by ECOG pairs and defines participant-grouped modeling helpers. Pair-level evaluation is primary; participant summaries exclude participants with changing labels.

In [16]:
# DATASET 4 PARALLEL EXPERIMENTS (17-26)
# Dataset 4 is prepared in the other notebook. The three-dataset
# analysis above is preserved unchanged. This extension runs after
# it, with separate variables, grouped participant CV, and outputs.
# Dataset 4 rows are repeated Storyteller-ECOG pairs. Pair-level
# metrics are primary; participant metrics exclude changing labels.
# ============================================================

from scipy.stats import spearmanr, pearsonr

D4_INPUT_FILE = DATA_DIR / "dataset4_multimodal_cn_vs_noncn.csv"
D4_RESULTS_DIR = RESULTS_ROOT / "dataset4_setup"
D4_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if not D4_INPUT_FILE.is_file():
    raise FileNotFoundError(
        "Prepare Dataset 4 in the other notebook first: " + str(D4_INPUT_FILE)
    )
dataset4_multimodal_cn_vs_noncn = pd.read_csv(D4_INPUT_FILE, low_memory=False)
D4_STORY = list(story_feature_cols)
D4_ECOG = list(ecog_feature_cols)
D4_FULL = D4_STORY + D4_ECOG
D4_ID, D4_Y = "PTID", "TARGET"
D4_SEEDS = [11, 21, 42, 52, 62, 72, 82, 92, 102, 112]
D4_PRIMARY, D4_K = 42, 5
d4_df = dataset4_multimodal_cn_vs_noncn.copy()
d4_missing = [c for c in [D4_ID, D4_Y, *D4_FULL] if c not in d4_df]
if d4_missing:
    raise KeyError(f"Dataset 4 is missing columns: {d4_missing}")
d4_df[D4_ID] = d4_df[D4_ID].astype("string").str.strip()
d4_df[D4_Y] = pd.to_numeric(d4_df[D4_Y], errors="coerce")
if d4_df[[D4_ID, D4_Y]].isna().any().any() or not d4_df[D4_Y].isin([0, 1]).all():
    raise ValueError("Dataset 4 needs nonmissing PTID and binary TARGET values.")
d4_df[D4_Y] = d4_df[D4_Y].astype(int)
for c in D4_FULL:
    d4_df[c] = pd.to_numeric(d4_df[c], errors="coerce")
for c in D4_ECOG:
    d4_df[c] = d4_df[c].replace(9, np.nan)
d4_y = d4_df[D4_Y].to_numpy()
d4_groups = d4_df[D4_ID].astype(str).to_numpy()
d4_label_counts = d4_df.groupby(D4_ID)[D4_Y].nunique()
d4_stable_ids = set(d4_label_counts[d4_label_counts == 1].index.astype(str))
d4_changing_ids = set(d4_label_counts[d4_label_counts > 1].index.astype(str))

print("\nDATASET 4 AUDIT")
print("Pair rows:", len(d4_df), "Participants:", d4_df[D4_ID].nunique())
print("Stable-label participants:", len(d4_stable_ids))
print("Changing-label participants:", len(d4_changing_ids))
print("Targets:", d4_df[D4_Y].value_counts().sort_index().to_dict())
print("D4 results directory:", D4_RESULTS_DIR)

def d4_model(seed):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=5000, class_weight="balanced",
            solver="liblinear", random_state=seed
        )),
    ])

def d4_metrics(y_true, probabilities):
    yt = np.asarray(y_true, dtype=int)
    pp = np.asarray(probabilities, dtype=float)
    yh = (pp >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
    return {
        "N": len(yt),
        "Accuracy": accuracy_score(yt, yh),
        "Balanced_Accuracy": balanced_accuracy_score(yt, yh),
        "MCI_Precision": precision_score(yt, yh, zero_division=0),
        "MCI_Recall": recall_score(yt, yh, zero_division=0),
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "MCI_F1": f1_score(yt, yh, zero_division=0),
        "Macro_F1": f1_score(yt, yh, average="macro", zero_division=0),
        "ROC_AUC": roc_auc_score(yt, pp) if len(np.unique(yt)) == 2 else np.nan,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }

def d4_folds(y, groups, seed, k=D4_K):
    cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    folds = list(cv.split(np.zeros(len(y)), y, groups))
    for tr, te in folds:
        if set(groups[tr]) & set(groups[te]):
            raise RuntimeError("Dataset 4 participant leakage detected.")
    return folds

def d4_oof(frame, features, seed, folds=None):
    yy = frame[D4_Y].to_numpy(dtype=int)
    gg = frame[D4_ID].astype(str).to_numpy()
    folds = d4_folds(yy, gg, seed) if folds is None else folds
    xx = frame[features].apply(pd.to_numeric, errors="coerce")
    pp = np.full(len(frame), np.nan)
    for tr, te in folds:
        m = d4_model(seed)
        m.fit(xx.iloc[tr], yy[tr])
        pp[te] = m.predict_proba(xx.iloc[te])[:, 1]
    if np.isnan(pp).any():
        raise AssertionError("Dataset 4 OOF predictions are incomplete.")
    return pp

def d4_participants(frame, probabilities):
    z = pd.DataFrame({
        "PTID": frame[D4_ID].astype(str).to_numpy(),
        "TARGET": frame[D4_Y].to_numpy(dtype=int),
        "PROBABILITY": np.asarray(probabilities),
    })
    counts = z.groupby("PTID")["TARGET"].nunique()
    keep = set(counts[counts == 1].index)
    return (z[z["PTID"].isin(keep)].groupby("PTID", as_index=False)
            .agg(TARGET=("TARGET", "first"),
                 PROBABILITY=("PROBABILITY", "mean"),
                 N_PAIRS=("TARGET", "size")))

def d4_evaluate(frame, specs, seed, exp_name):
    frame = frame.reset_index(drop=True).copy()
    yy = frame[D4_Y].to_numpy(dtype=int)
    gg = frame[D4_ID].astype(str).to_numpy()
    folds = d4_folds(yy, gg, seed)
    rows, oofs = [], {}
    for name, features in specs.items():
        pp = d4_oof(frame, features, seed, folds)
        oofs[name] = pp
        part = d4_participants(frame, pp)
        rows.append({
            "Experiment": exp_name, "Seed": seed, "Model": name,
            "Evaluation_unit": "Pair", **d4_metrics(yy, pp)
        })
        rows.append({
            "Experiment": exp_name, "Seed": seed, "Model": name,
            "Evaluation_unit": "Stable-label participant",
            **d4_metrics(part["TARGET"], part["PROBABILITY"])
        })
    return pd.DataFrame(rows), oofs

D4_MODALITIES = {
    "Storyteller only": D4_STORY,
    "ECOG only": D4_ECOG,
    "Full fusion": D4_FULL,
}




DATASET 4 AUDIT
Pair rows: 877 Participants: 221
Stable-label participants: 218
Changing-label participants: 3
Targets: {0: 598, 1: 279}
D4 results directory: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\results\dataset4_setup


### Dataset 4: All-Pairs Multimodal Baseline

Results are saved in results/historical_prior_ecog_classification/dataset4/.

In [17]:
# Historical Prior ECOG Multimodal Classification baseline modality models on the prepared all-pairs cohort.
D4_RESULTS_DIR=RESULTS_ROOT / "historical_prior_ecog_classification" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
d4_exp17, d4_exp17_oof = d4_evaluate(
    d4_df, D4_MODALITIES, D4_PRIMARY, "Experiment 17"
)
display(d4_exp17.round(4))
d4_exp17.to_csv(D4_RESULTS_DIR / "dataset4_historical_prior_ecog_classification_metrics.csv", index=False)
pd.DataFrame({"PTID": d4_df[D4_ID], "TARGET": d4_y, **d4_exp17_oof}).to_csv(
    D4_RESULTS_DIR / "dataset4_historical_prior_ecog_classification_oof.csv", index=False
)



,Experiment,Seed,Model,Evaluation_unit,N,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,Specificity,MCI_F1,Macro_F1,ROC_AUC,TN,FP,FN,TP
0,Experiment 17,42,Storyteller only,Pair,877,0.6499,0.6190,0.4571,0.5341,0.7040,0.4926,0.6127,0.6577,421,177,130,149
1,Experiment 17,42,Storyteller only,Stable-label participant,218,0.6835,0.6744,0.4831,0.6515,0.6974,0.5548,0.6546,0.7095,106,46,23,43
2,Experiment 17,42,ECOG only,Pair,877,0.6933,0.6527,0.5171,0.5412,0.7642,0.5289,0.6508,0.6517,457,141,128,151
3,Experiment 17,42,ECOG only,Stable-label participant,218,0.7018,0.6662,0.5067,0.5758,0.7566,0.5390,0.6593,0.6756,115,37,28,38
4,Experiment 17,42,Full fusion,Pair,877,0.6830,0.6481,0.5016,0.5520,0.7441,0.5256,0.6438,0.6949,445,153,125,154
5,Experiment 17,42,Full fusion,Stable-label participant,218,0.6789,0.6540,0.4756,0.5909,0.7171,0.5270,0.6420,0.7360,109,43,27,39


### Dataset 4: Temporal Window Sensitivity

Results are saved in results/temporal_window_sensitivity/dataset4/.

In [18]:
# Temporal Window Sensitivity and Fixed Cohort Comparison pair-date sensitivity. E18 uses each eligible window
D4_RESULTS_DIR=RESULTS_ROOT / "temporal_window_sensitivity" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
# cohort. E19 holds fixed Storyteller assessments eligible inside 30 days,
# while the prior-ECOG pair window expands. Dataset 4 visit labels can vary,
# so E19 pair rows are not fixed even though its Storyteller cohort is fixed.
d4_story_date = next(
    (c for c in ("EXAMDATE_STORY", "STORY_DATE") if c in d4_df), None
)
d4_ecog_date = next(
    (c for c in ("COLDATE", "COLDATE_ECOG", "EXAMDATE_ECOG") if c in d4_df),
    None
)
d4_temp_rows, d4_temp_audit = [], []
if d4_story_date and d4_ecog_date:
    d4_temp = d4_df.copy()
    d4_temp[d4_story_date] = pd.to_datetime(d4_temp[d4_story_date], errors="coerce")
    d4_temp[d4_ecog_date] = pd.to_datetime(d4_temp[d4_ecog_date], errors="coerce")
    d4_temp["DAYS_ECOG_BEFORE_STORY"] = (
        d4_temp[d4_story_date] - d4_temp[d4_ecog_date]
    ).dt.days
    if "_D4_STORY_ROW_ID" in d4_temp:
        d4_story_key = "_D4_STORY_ROW_ID"
    elif {"ADNIOnlineID", "RMT_Timepoint"}.issubset(d4_temp.columns):
        d4_story_key = "_D4_STORY_KEY"
        d4_temp[d4_story_key] = (
            d4_temp["PTID"].astype(str) + "|" +
            d4_temp["ADNIOnlineID"].astype(str) + "|" +
            d4_temp["RMT_Timepoint"].astype(str)
        )
    else:
        d4_story_key = "_D4_STORY_KEY"
        d4_temp[d4_story_key] = (
            d4_temp["PTID"].astype(str) + "|" +
            d4_temp[d4_story_date].astype(str)
        )
    d4_windows = {
        "30_days": 30, "90_days": 90, "180_days": 180,
        "365_days": 365, "historical": None,
    }
    d4_fixed_story = set(d4_temp.loc[
        d4_temp["DAYS_ECOG_BEFORE_STORY"].between(0, 30), d4_story_key
    ].dropna())
    for exp_no in (18, 19):
        for win_name, max_days in d4_windows.items():
            mask = d4_temp["DAYS_ECOG_BEFORE_STORY"].ge(0)
            if max_days is not None:
                mask &= d4_temp["DAYS_ECOG_BEFORE_STORY"].le(max_days)
            if exp_no == 19:
                mask &= d4_temp[d4_story_key].isin(d4_fixed_story)
            subset = d4_temp.loc[mask].copy()
            if subset.empty:
                continue
            try:
                res, _ = d4_evaluate(
                    subset, D4_MODALITIES, D4_PRIMARY,
                    f"Experiment {exp_no} - {win_name}"
                )
                d4_temp_rows.append(res)
                d4_temp_audit.append({
                    "Experiment": exp_no, "Window": win_name,
                    "Pair_rows": len(subset),
                    "Participants": subset["PTID"].nunique(),
                    "Storyteller_assessments": subset[d4_story_key].nunique(),
                })
            except ValueError as exc:
                print(f"D4 experiment {exp_no}, {win_name} skipped: {exc}")
else:
    print("Dataset 4 Experiments 18-19 skipped: pair date columns are absent.")
if d4_temp_rows:
    d4_temp_results = pd.concat(d4_temp_rows, ignore_index=True)
    d4_temp_audit_df = pd.DataFrame(d4_temp_audit)
    display(d4_temp_audit_df)
    display(d4_temp_results.round(4))
    d4_temp_results.to_csv(
        D4_RESULTS_DIR / "dataset4_temporal_window_metrics.csv", index=False
    )
    d4_temp_audit_df.to_csv(
        D4_RESULTS_DIR / "dataset4_temporal_window_audit.csv", index=False
    )



,Experiment,Window,Pair_rows,Participants,Storyteller_assessments
0,18,30_days,77,71,77
1,18,90_days,150,132,150
2,18,180_days,183,160,183
3,18,365_days,329,175,328
4,18,historical,549,185,515
5,19,30_days,77,71,77
6,19,90_days,77,71,77
7,19,180_days,77,71,77
8,19,365_days,78,71,77
9,19,historical,87,71,77


,Experiment,Seed,Model,Evaluation_unit,N,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,Specificity,MCI_F1,Macro_F1,ROC_AUC,TN,FP,FN,TP
0,Experiment 18 - 30_days,42,Storyteller only,Pair,77,0.6623,0.6065,0.4583,0.4583,0.7547,0.4583,0.6065,0.6627,40,13,13,11
1,Experiment 18 - 30_days,42,Storyteller only,Stable-label participant,71,0.6479,0.5946,0.4348,0.4545,0.7347,0.4444,0.5934,0.6670,36,13,12,10
2,Experiment 18 - 30_days,42,ECOG only,Pair,77,0.7532,0.7068,0.6087,0.5833,0.8302,0.5957,0.7091,0.6895,44,9,10,14
3,Experiment 18 - 30_days,42,ECOG only,Stable-label participant,71,0.7465,0.7036,0.5909,0.5909,0.8163,0.5909,0.7036,0.6892,40,9,9,13
4,Experiment 18 - 30_days,42,Full fusion,Pair,77,0.7662,0.7390,0.6154,0.6667,0.8113,0.6400,0.7335,0.7972,43,10,8,16
5,Experiment 18 - 30_days,42,Full fusion,Stable-label participant,71,0.7746,0.7616,0.6154,0.7273,0.7959,0.6667,0.7482,0.8071,39,10,6,16
6,Experiment 18 - 90_days,42,Storyteller only,Pair,150,0.5800,0.5423,0.3684,0.4375,0.6471,0.4000,0.5385,0.5746,66,36,27,21
7,Experiment 18 - 90_days,42,Storyteller only,Stable-label participant,132,0.5909,0.5608,0.3585,0.4872,0.6344,0.4130,0.5495,0.6118,59,34,20,19
8,Experiment 18 - 90_days,42,ECOG only,Pair,150,0.7400,0.6930,0.6000,0.5625,0.8235,0.5806,0.6961,0.7412,84,18,21,27
9,Experiment 18 - 90_days,42,ECOG only,Stable-label participant,132,0.7348,0.6853,0.5500,0.5641,0.8065,0.5570,0.6839,0.7589,75,18,17,22


### Dataset 4: ECOG Item Ablation

Results are saved in results/ecog_item_ablation/dataset4/.

In [19]:
# ECOG Item Ablation Analysis leave-one-ECOG-item-out ablation.
D4_RESULTS_DIR=RESULTS_ROOT / "ecog_item_ablation" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
D4_ABLATIONS = dict(D4_MODALITIES)
for item in D4_ECOG:
    D4_ABLATIONS[f"Fusion without {item}"] = [
        c for c in D4_FULL if c != item
    ]
d4_exp20, d4_exp20_oof = d4_evaluate(
    d4_df, D4_ABLATIONS, D4_PRIMARY, "Experiment 20"
)
display(d4_exp20.round(4))
d4_exp20.to_csv(D4_RESULTS_DIR / "dataset4_ecog_item_ablation_ablation.csv", index=False)



,Experiment,Seed,Model,Evaluation_unit,N,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,Specificity,MCI_F1,Macro_F1,ROC_AUC,TN,FP,FN,TP
0,Experiment 20,42,Storyteller only,Pair,877,0.6499,0.6190,0.4571,0.5341,0.7040,0.4926,0.6127,0.6577,421,177,130,149
1,Experiment 20,42,Storyteller only,Stable-label participant,218,0.6835,0.6744,0.4831,0.6515,0.6974,0.5548,0.6546,0.7095,106,46,23,43
2,Experiment 20,42,ECOG only,Pair,877,0.6933,0.6527,0.5171,0.5412,0.7642,0.5289,0.6508,0.6517,457,141,128,151
3,Experiment 20,42,ECOG only,Stable-label participant,218,0.7018,0.6662,0.5067,0.5758,0.7566,0.5390,0.6593,0.6756,115,37,28,38
4,Experiment 20,42,Full fusion,Pair,877,0.6830,0.6481,0.5016,0.5520,0.7441,0.5256,0.6438,0.6949,445,153,125,154
5,Experiment 20,42,Full fusion,Stable-label participant,218,0.6789,0.6540,0.4756,0.5909,0.7171,0.5270,0.6420,0.7360,109,43,27,39
6,Experiment 20,42,Fusion without ECOGPT1,Pair,877,0.6944,0.6622,0.5178,0.5735,0.7508,0.5442,0.6572,0.7092,449,149,119,160
7,Experiment 20,42,Fusion without ECOGPT1,Stable-label participant,218,0.7064,0.6952,0.5116,0.6667,0.7237,0.5789,0.6768,0.7442,110,42,22,44
8,Experiment 20,42,Fusion without ECOGPT2,Pair,877,0.6899,0.6550,0.5115,0.5591,0.7508,0.5342,0.6509,0.6974,449,149,123,156
9,Experiment 20,42,Fusion without ECOGPT2,Stable-label participant,218,0.7018,0.6747,0.5063,0.6061,0.7434,0.5517,0.6642,0.7456,113,39,26,40


### Dataset 4: Repeated Seed Ablation Stability

Results are saved in results/ecog_ablation_stability/dataset4/.

In [20]:
# Repeated Seed Stability of ECOG Item Ablations preselected ECOG item sensitivity across the same seed set.
D4_RESULTS_DIR=RESULTS_ROOT / "ecog_ablation_stability" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
D4_STABILITY = dict(D4_MODALITIES)
for item in ("ECOGPT1", "ECOGPT2", "ECOGPT8", "ECOGPT9"):
    D4_STABILITY[f"Fusion without {item}"] = [c for c in D4_FULL if c != item]
d4_exp21 = pd.concat([
    d4_evaluate(d4_df, D4_STABILITY, seed, "Experiment 21")[0]
    for seed in D4_SEEDS
], ignore_index=True)
d4_exp21_summary = d4_exp21.groupby(
    ["Model", "Evaluation_unit"]
)[["Balanced_Accuracy", "MCI_Recall", "Specificity", "MCI_F1", "ROC_AUC"]].agg(
    ["mean", "std"]
)
display(d4_exp21_summary.round(4))
d4_exp21.to_csv(D4_RESULTS_DIR / "dataset4_ecog_ablation_stability_all_seeds.csv", index=False)
d4_exp21_summary.to_csv(D4_RESULTS_DIR / "dataset4_ecog_ablation_stability_stability.csv")



Balanced_Accuracy          \
                                                             mean     std   
Model                  Evaluation_unit                                      
ECOG only              Pair                                0.6469  0.0195   
                       Stable-label participant            0.6571  0.0213   
Full fusion            Pair                                0.6497  0.0222   
                       Stable-label participant            0.6724  0.0220   
Fusion without ECOGPT1 Pair                                0.6573  0.0233   
                       Stable-label participant            0.6852  0.0226   
Fusion without ECOGPT2 Pair                                0.6520  0.0205   
                       Stable-label participant            0.6712  0.0186   
Fusion without ECOGPT8 Pair                                0.6406  0.0161   
                       Stable-label participant            0.6583  0.0195   
Fusion without ECOGPT9 Pair                                0.6523  0.0197   
                       Stable-label participant            0.6719  0.0215   
Storyteller only       Pair                                0.6200  0.0121   
                       Stable-label participant            0.6670  0.0242   

                                                MCI_Recall          \
                                                      mean     std   
Model                  Evaluation_unit                               
ECOG only              Pair                         0.5290  0.0359   
                       Stable-label participant     0.5576  0.0383   
Full fusion            Pair                         0.5487  0.0349   
                       Stable-label participant     0.6106  0.0371   
Fusion without ECOGPT1 Pair                         0.5638  0.0364   
                       Stable-label participant     0.6348  0.0360   
Fusion without ECOGPT2 Pair                         0.5505  0.0322   
                       Stable-label participant     0.6076  0.0323   
Fusion without ECOGPT8 Pair                         0.5477  0.0250   
                       Stable-label participant     0.5864  0.0343   
Fusion without ECOGPT9 Pair                         0.5534  0.0338   
                       Stable-label participant     0.6121  0.0321   
Storyteller only       Pair                         0.5530  0.0216   
                       Stable-label participant     0.6379  0.0394   

                                                Specificity          MCI_F1  \
                                                       mean     std    mean   
Model                  Evaluation_unit                                        
ECOG only              Pair                          0.7647  0.0176  0.5201   
                       Stable-label participant      0.7566  0.0167  0.5262   
Full fusion            Pair                          0.7507  0.0181  0.5267   
                       Stable-label participant      0.7342  0.0181  0.5493   
Fusion without ECOGPT1 Pair                          0.7508  0.0169  0.5374   
                       Stable-label participant      0.7355  0.0169  0.5657   
Fusion without ECOGPT2 Pair                          0.7535  0.0172  0.5296   
                       Stable-label participant      0.7349  0.0135  0.5477   
Fusion without ECOGPT8 Pair                          0.7336  0.0142  0.5170   
                       Stable-label participant      0.7303  0.0155  0.5311   
Fusion without ECOGPT9 Pair                          0.7512  0.0160  0.5302   
                       Stable-label participant      0.7316  0.0188  0.5489   
Storyteller only       Pair                          0.6870  0.0160  0.4973   
                       Stable-label participant      0.6961  0.0232  0.5456   

                                                        ROC_AUC          
                                                    std    mean     std  
Model                  Evaluation_unit                 

### Dataset 4: Nested Feature Selection

Results are saved in results/nested_feature_selection/dataset4/.

In [21]:
# Nested Participant Grouped Feature Selection nested selection among the same four ECOG configurations.
D4_RESULTS_DIR=RESULTS_ROOT / "nested_feature_selection" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
D4_CANDIDATES = {
    "Full fusion": D4_FULL,
    "Without ECOGPT1": [c for c in D4_FULL if c != "ECOGPT1"],
    "Without ECOGPT2": [c for c in D4_FULL if c != "ECOGPT2"],
    "Without ECOGPT1 and ECOGPT2": [
        c for c in D4_FULL if c not in ("ECOGPT1", "ECOGPT2")
    ],
}
d4_exp22_rows = []
for seed in D4_SEEDS:
    outer = d4_folds(d4_y, d4_groups, seed)
    oof = np.full(len(d4_df), np.nan)
    selected_by_fold = []
    for outer_no, (otr, ote) in enumerate(outer, 1):
        iy, ig = d4_y[otr], d4_groups[otr]
        inner_cv = StratifiedGroupKFold(
            n_splits=4, shuffle=True, random_state=seed + outer_no
        )
        inner = list(inner_cv.split(np.zeros(len(iy)), iy, ig))
        scores = {}
        for candidate, features in D4_CANDIDATES.items():
            inner_oof = np.full(len(otr), np.nan)
            X = d4_df.iloc[otr][features]
            for itr, iva in inner:
                if set(ig[itr]) & set(ig[iva]):
                    raise RuntimeError("D4 nested inner-fold PTID leakage.")
                model = d4_model(seed)
                model.fit(X.iloc[itr], iy[itr])
                inner_oof[iva] = model.predict_proba(X.iloc[iva])[:, 1]
            scores[candidate] = roc_auc_score(iy, inner_oof)
        chosen = max(scores, key=scores.get)
        selected_by_fold.append(chosen)
        model = d4_model(seed)
        model.fit(d4_df.iloc[otr][D4_CANDIDATES[chosen]], d4_y[otr])
        oof[ote] = model.predict_proba(
            d4_df.iloc[ote][D4_CANDIDATES[chosen]]
        )[:, 1]
    part = d4_participants(d4_df, oof)
    for unit, values in (
        ("Pair", d4_metrics(d4_y, oof)),
        ("Stable-label participant", d4_metrics(part["TARGET"], part["PROBABILITY"])),
    ):
        d4_exp22_rows.append({
            "Seed": seed, "Evaluation_unit": unit, **values,
            "Outer_folds_selected_without_item1": sum(
                ("ECOGPT1" not in D4_CANDIDATES[n]) for n in selected_by_fold
            ),
            "Outer_folds_selected_without_item2": sum(
                ("ECOGPT2" not in D4_CANDIDATES[n]) for n in selected_by_fold
            ),
        })
d4_exp22 = pd.DataFrame(d4_exp22_rows)
display(d4_exp22.round(4))
d4_exp22.to_csv(D4_RESULTS_DIR / "dataset4_nested_feature_selection_nested_selection.csv", index=False)



,Seed,Evaluation_unit,N,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,Specificity,MCI_F1,Macro_F1,ROC_AUC,TN,FP,FN,TP,Outer_folds_selected_without_item1,Outer_folds_selected_without_item2
0,11,Pair,877,0.7092,0.6730,0.5405,0.5735,0.7726,0.5565,0.6701,0.7184,462,136,119,160,4,4
1,11,Stable-label participant,218,0.7110,0.6856,0.5190,0.6212,0.7500,0.5655,0.6745,0.7557,114,38,25,41,4,4
2,21,Pair,877,0.6842,0.6355,0.5036,0.5018,0.7692,0.5027,0.6356,0.6813,460,138,139,140,5,4
3,21,Stable-label participant,218,0.7110,0.6813,0.5195,0.6061,0.7566,0.5594,0.6722,0.7089,115,37,26,40,5,4
4,42,Pair,877,0.6990,0.6674,0.5243,0.5806,0.7542,0.5510,0.6623,0.7116,451,147,117,162,5,5
5,42,Stable-label participant,218,0.7018,0.6876,0.5059,0.6515,0.7237,0.5695,0.6707,0.7530,110,42,23,43,5,5
6,52,Pair,877,0.6556,0.6184,0.4630,0.5161,0.7207,0.4881,0.6143,0.6537,431,167,135,144,5,4
7,52,Stable-label participant,218,0.6835,0.6573,0.4815,0.5909,0.7237,0.5306,0.6459,0.7026,110,42,27,39,5,4
8,62,Pair,877,0.6659,0.6260,0.4768,0.5161,0.7358,0.4957,0.6230,0.6502,440,158,135,144,4,3
9,62,Stable-label participant,218,0.6789,0.6411,0.4737,0.5455,0.7368,0.5070,0.6345,0.6789,112,40,30,36,4,3


### Dataset 4: Final Model Comparison and Bootstrap Uncertainty

Results are saved in results/final_model_comparison_uncertainty/dataset4/.

In [22]:
# Final Model Comparison and Uncertainty Analysis final fixed model set, repeated seeds, paired participant bootstrap.
D4_RESULTS_DIR=RESULTS_ROOT / "final_model_comparison_uncertainty" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
D4_FINAL = {
    "Storyteller only": D4_STORY,
    "Full fusion": D4_FULL,
    "Fusion without ECOGPT1": [c for c in D4_FULL if c != "ECOGPT1"],
    "Fusion without ECOGPT1 and ECOGPT2": [
        c for c in D4_FULL if c not in ("ECOGPT1", "ECOGPT2")
    ],
    "Fusion without ECOGPT2": [c for c in D4_FULL if c != "ECOGPT2"],
}
d4_final_rows, d4_seed42_oof = [], {}
for seed in D4_SEEDS:
    res, oofs = d4_evaluate(d4_df, D4_FINAL, seed, "Experiment 23")
    d4_final_rows.append(res)
    if seed == D4_PRIMARY:
        d4_seed42_oof = oofs
d4_exp23 = pd.concat(d4_final_rows, ignore_index=True)
d4_exp23_stability = d4_exp23.groupby(
    ["Model", "Evaluation_unit"]
)[["Accuracy", "Balanced_Accuracy", "MCI_Recall", "Specificity", "MCI_F1", "ROC_AUC"]].agg(
    ["mean", "std"]
)
display(d4_exp23[d4_exp23["Seed"] == D4_PRIMARY].round(4))
display(d4_exp23_stability.round(4))
d4_exp23.to_csv(D4_RESULTS_DIR / "dataset4_final_model_comparison_uncertainty_all_seeds.csv", index=False)
d4_exp23_stability.to_csv(D4_RESULTS_DIR / "dataset4_final_model_comparison_uncertainty_stability.csv")

d4_boot = pd.DataFrame({"PTID": d4_groups, "TARGET": d4_y, **d4_seed42_oof})
d4_boot = d4_boot[d4_boot["PTID"].isin(d4_stable_ids)].groupby(
    "PTID", as_index=False
).agg({"TARGET": "first", **{m: "mean" for m in D4_FINAL}})
rng = np.random.default_rng(42023)
boot_rows = []
for comparison in D4_FINAL:
    if comparison == "Full fusion":
        continue
    delta = []
    for _ in range(2000):
        sample = d4_boot.iloc[rng.integers(0, len(d4_boot), len(d4_boot))]
        if sample["TARGET"].nunique() == 2:
            delta.append(
                roc_auc_score(sample["TARGET"], sample[comparison])
                - roc_auc_score(sample["TARGET"], sample["Full fusion"])
            )
    boot_rows.append({
        "Reference": "Full fusion", "Comparison": comparison,
        "AUC_Difference": (
            roc_auc_score(d4_boot["TARGET"], d4_boot[comparison])
            - roc_auc_score(d4_boot["TARGET"], d4_boot["Full fusion"])
        ),
        "CI_Lower": np.quantile(delta, 0.025),
        "CI_Upper": np.quantile(delta, 0.975),
        "Valid_Replicates": len(delta),
    })
d4_bootstrap = pd.DataFrame(boot_rows)
display(d4_bootstrap.round(4))
d4_bootstrap.to_csv(D4_RESULTS_DIR / "dataset4_final_model_comparison_uncertainty_bootstrap.csv", index=False)



,Experiment,Seed,Model,Evaluation_unit,N,Accuracy,Balanced_Accuracy,MCI_Precision,MCI_Recall,Specificity,MCI_F1,Macro_F1,ROC_AUC,TN,FP,FN,TP
20,Experiment 23,42,Storyteller only,Pair,877,0.6499,0.6190,0.4571,0.5341,0.7040,0.4926,0.6127,0.6577,421,177,130,149
21,Experiment 23,42,Storyteller only,Stable-label participant,218,0.6835,0.6744,0.4831,0.6515,0.6974,0.5548,0.6546,0.7095,106,46,23,43
22,Experiment 23,42,Full fusion,Pair,877,0.6830,0.6481,0.5016,0.5520,0.7441,0.5256,0.6438,0.6949,445,153,125,154
23,Experiment 23,42,Full fusion,Stable-label participant,218,0.6789,0.6540,0.4756,0.5909,0.7171,0.5270,0.6420,0.7360,109,43,27,39
24,Experiment 23,42,Fusion without ECOGPT1,Pair,877,0.6944,0.6622,0.5178,0.5735,0.7508,0.5442,0.6572,0.7092,449,149,119,160
25,Experiment 23,42,Fusion without ECOGPT1,Stable-label participant,218,0.7064,0.6952,0.5116,0.6667,0.7237,0.5789,0.6768,0.7442,110,42,22,44
26,Experiment 23,42,Fusion without ECOGPT1 and ECOGPT2,Pair,877,0.6990,0.6674,0.5243,0.5806,0.7542,0.5510,0.6623,0.7116,451,147,117,162
27,Experiment 23,42,Fusion without ECOGPT1 and ECOGPT2,Stable-label participant,218,0.7018,0.6876,0.5059,0.6515,0.7237,0.5695,0.6707,0.7530,110,42,23,43
28,Experiment 23,42,Fusion without ECOGPT2,Pair,877,0.6899,0.6550,0.5115,0.5591,0.7508,0.5342,0.6509,0.6974,449,149,123,156
29,Experiment 23,42,Fusion without ECOGPT2,Stable-label participant,218,0.7018,0.6747,0.5063,0.6061,0.7434,0.5517,0.6642,0.7456,113,39,26,40


Accuracy          \
                                                                mean     std   
Model                              Evaluation_unit                             
Full fusion                        Pair                       0.6864  0.0192   
                                   Stable-label participant   0.6968  0.0183   
Fusion without ECOGPT1             Pair                       0.6913  0.0197   
                                   Stable-label participant   0.7050  0.0189   
Fusion without ECOGPT1 and ECOGPT2 Pair                       0.6945  0.0176   
                                   Stable-label participant   0.7055  0.0124   
Fusion without ECOGPT2             Pair                       0.6889  0.0178   
                                   Stable-label participant   0.6963  0.0148   
Storyteller only                   Pair                       0.6444  0.0117   
                                   Stable-label participant   0.6784  0.0213   

                                                            Balanced_Accuracy  \
                                                                         mean   
Model                              Evaluation_unit                              
Full fusion                        Pair                                0.6497   
                                   Stable-label participant            0.6724   
Fusion without ECOGPT1             Pair                                0.6573   
                                   Stable-label participant            0.6852   
Fusion without ECOGPT1 and ECOGPT2 Pair                                0.6598   
                                   Stable-label participant            0.6829   
Fusion without ECOGPT2             Pair                                0.6520   
                                   Stable-label participant            0.6712   
Storyteller only                   Pair                                0.6200   
                                   Stable-label participant            0.6670   

                                                                     \
                                                                std   
Model                              Evaluation_unit                    
Full fusion                        Pair                      0.0222   
                                   Stable-label participant  0.0220   
Fusion without ECOGPT1             Pair                      0.0233   
                                   Stable-label participant  0.0226   
Fusion without ECOGPT1 and ECOGPT2 Pair                      0.0214   
                                   Stable-label participant  0.0161   
Fusion without ECOGPT2             Pair                      0.0205   
                                   Stable-label participant  0.0186   
Storyteller only                   Pair                      0.0121   
                                   Stable-label participant  0.0242   

                                                            MCI_Recall  \
                                                                  mean   
Model                              Evaluation_unit                       
Full fusion                        Pair                         0.5487   
                                   Stable-label participant     0.6106   
Fusion without ECOGPT1             Pair                         0.5638   
                                   Stable-label participant     0.6348   
Fusion without ECOGPT1 and ECOGPT2 Pair                         0.5642   
                                   Stable-label participant     0.6258   
Fusion without ECOGPT2             Pair                         0.5505   
                                   Stable-label participant     0.6076   
Storyteller only                   Pair                         0.5530   
                                   Stable-label participant     0.6379   

                                                                     \
       

,Reference,Comparison,AUC_Difference,CI_Lower,CI_Upper,Valid_Replicates
0,Full fusion,Storyteller only,-0.0265,-0.0902,0.0382,2000
1,Full fusion,Fusion without ECOGPT1,0.0082,-0.0035,0.0207,2000
2,Full fusion,Fusion without ECOGPT1 and ECOGPT2,0.0169,0.0036,0.0310,2000
3,Full fusion,Fusion without ECOGPT2,0.0096,0.0026,0.0176,2000


### Dataset 4: Modality and Feature Interpretation

Results are saved in results/feature_interpretation/dataset4/.

In [23]:
# Modality and Feature Interpretation participant-median features on stable-label participants.
D4_RESULTS_DIR=RESULTS_ROOT / "feature_interpretation" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
d4_stable_rows = d4_df[d4_df[D4_ID].astype(str).isin(d4_stable_ids)]
d4_part_X = d4_stable_rows.groupby(D4_ID)[D4_FULL].median()
d4_part_y = d4_stable_rows.groupby(D4_ID)[D4_Y].first().reindex(d4_part_X.index).astype(int)
d4_part_groups = d4_part_X.index.astype(str).to_numpy()
D4_PART_MODELS = {
    "Storyteller only": D4_STORY,
    "ECOG only (items 3-12)": [c for c in D4_ECOG if c not in ("ECOGPT1", "ECOGPT2")],
    "Full fusion": D4_FULL,
    "Reduced fusion (without ECOGPT1 and ECOGPT2)": [
        c for c in D4_FULL if c not in ("ECOGPT1", "ECOGPT2")
    ],
}
interpretation_rows, permutation_rows, coefficient_rows = [], [], []
for seed in D4_SEEDS:
    folds = d4_folds(d4_part_y.to_numpy(), d4_part_groups, seed)
    for name, features in D4_PART_MODELS.items():
        oof = np.full(len(d4_part_X), np.nan)
        for fold_no, (tr, te) in enumerate(folds, 1):
            Xtr, Xte = d4_part_X.iloc[tr][features], d4_part_X.iloc[te][features]
            ytr, yte = d4_part_y.iloc[tr].to_numpy(), d4_part_y.iloc[te].to_numpy()
            model = d4_model(seed)
            model.fit(Xtr, ytr)
            p = model.predict_proba(Xte)[:, 1]
            oof[te] = p
            if name == "Reduced fusion (without ECOGPT1 and ECOGPT2)":
                base_auc = roc_auc_score(yte, p)
                for feature in features:
                    permuted = Xte.copy()
                    permuted[feature] = np.random.default_rng(
                        seed * 1000 + fold_no
                    ).permutation(permuted[feature].to_numpy())
                    perm_auc = roc_auc_score(yte, model.predict_proba(permuted)[:, 1])
                    permutation_rows.append({
                        "Seed": seed, "Fold": fold_no, "Feature": feature,
                        "Baseline_AUC": base_auc, "AUC_Drop": base_auc - perm_auc,
                    })
                feature_names = model.named_steps["imputer"].get_feature_names_out(features)
                for feature, coef in zip(
                    feature_names, model.named_steps["classifier"].coef_[0]
                ):
                    coefficient_rows.append({
                        "Seed": seed, "Fold": fold_no, "Feature": feature,
                        "Coefficient": coef, "Absolute_Coefficient": abs(coef),
                    })
        interpretation_rows.append({
            "Seed": seed, "Model": name,
            **d4_metrics(d4_part_y.to_numpy(), oof),
        })

d4_exp24 = pd.DataFrame(interpretation_rows)
d4_exp24_stability = d4_exp24.groupby("Model")[
    ["Balanced_Accuracy", "MCI_Recall", "Specificity", "MCI_F1", "ROC_AUC"]
].agg(["mean", "std"])
d4_perm = pd.DataFrame(permutation_rows)
d4_perm_summary = d4_perm.groupby("Feature").agg(
    Mean_AUC_Drop=("AUC_Drop", "mean"),
    SD_AUC_Drop=("AUC_Drop", "std"),
    Positive_Importance_Frequency=("AUC_Drop", lambda x: (x > 0).mean()),
    N_Results=("AUC_Drop", "size"),
).sort_values("Mean_AUC_Drop", ascending=False)
d4_coef = pd.DataFrame(coefficient_rows)
d4_coef_summary = d4_coef.groupby("Feature").agg(
    Mean_Coefficient=("Coefficient", "mean"),
    SD_Coefficient=("Coefficient", "std"),
    Mean_Absolute_Coefficient=("Absolute_Coefficient", "mean"),
).sort_values("Mean_Absolute_Coefficient", ascending=False)
display(d4_exp24_stability.round(4))
display(d4_perm_summary.head(20).round(4))
display(d4_coef_summary.head(20).round(4))
d4_exp24.to_csv(D4_RESULTS_DIR / "dataset4_feature_interpretation_modality.csv", index=False)
d4_exp24_stability.to_csv(D4_RESULTS_DIR / "dataset4_feature_interpretation_stability.csv")
d4_perm.to_csv(D4_RESULTS_DIR / "dataset4_feature_interpretation_permutation_folds.csv", index=False)
d4_perm_summary.to_csv(D4_RESULTS_DIR / "dataset4_feature_interpretation_permutation_summary.csv")
d4_coef.to_csv(D4_RESULTS_DIR / "dataset4_feature_interpretation_coefficients.csv", index=False)
d4_coef_summary.to_csv(D4_RESULTS_DIR / "dataset4_feature_interpretation_coefficient_summary.csv")



Balanced_Accuracy          \
                                                          mean     std   
Model                                                                    
ECOG only (items 3-12)                                  0.6953  0.0157   
Full fusion                                             0.6757  0.0229   
Reduced fusion (without ECOGPT1 and ECOGPT2)            0.6878  0.0181   
Storyteller only                                        0.6505  0.0206   

                                             MCI_Recall         Specificity  \
                                                   mean     std        mean   
Model                                                                         
ECOG only (items 3-12)                           0.5939  0.0199      0.7967   
Full fusion                                      0.6000  0.0418      0.7513   
Reduced fusion (without ECOGPT1 and ECOGPT2)     0.6197  0.0331      0.7559   
Storyteller only                                 0.5924  0.0307      0.7086   

                                                      MCI_F1         ROC_AUC  \
                                                 std    mean     std    mean   
Model                                                                          
ECOG only (items 3-12)                        0.0153  0.5762  0.0209  0.7269   
Full fusion                                   0.0208  0.5520  0.0304  0.7259   
Reduced fusion (without ECOGPT1 and ECOGPT2)  0.0209  0.5680  0.0233  0.7386   
Storyteller only                              0.0167  0.5234  0.0253  0.6836   

                                                      
                                                 std  
Model                                                 
ECOG only (items 3-12)                        0.0122  
Full fusion                                   0.0231  
Reduced fusion (without ECOGPT1 and ECOGPT2)  0.0186  
Storyteller only                              0.0197

,Mean_AUC_Drop,SD_AUC_Drop,Positive_Importance_Frequency,N_Results
Feature,,,,
STORY1DEL_NUM_NOUNS,0.0866,0.0710,0.92,50
ECOGPT8,0.0809,0.0479,0.96,50
STORY1DEL_UNIQUE_WORDS_RATIO,0.0641,0.0652,0.90,50
STORY1IMM_NUM_PRONOUNS,0.0413,0.0366,0.90,50
STORY1DEL_ARTICULATION_RATE,0.0336,0.0475,0.76,50
STORY1IMM_NUM_ADJECTIVES,0.0325,0.0334,0.86,50
STORY1IMM_NUM_NOUNS,0.0317,0.0328,0.88,50
STORY1IMM_NUM_VERBS,0.0256,0.0362,0.84,50
STORY2IMM_NUM_NOUNS,0.0220,0.0385,0.80,50


,Mean_Coefficient,SD_Coefficient,Mean_Absolute_Coefficient
Feature,,,
STORY1IMM_UNIQUE_WORDS_RATIO,-1.1132,0.1896,1.1132
ECOGPT8,0.9924,0.2081,0.9924
STORY1DEL_UNIQUE_WORDS_RATIO,0.9520,0.2425,0.9520
STORY1DEL_NUM_NOUNS,-0.9249,0.3251,0.9329
STORY1DEL_ARTICULATION_RATE,0.8391,0.2418,0.8391
STORY1DEL_NUM_VERBS,0.7155,0.3466,0.7175
STORY2IMM_IDEA_DENSITY,-0.6633,0.1788,0.6647
STORY1IMM_NUM_PRONOUNS,-0.6169,0.2304,0.6169
STORY1IMM_NUM_ADJECTIVES,-0.5685,0.1822,0.5685


### Dataset 4: Feature Importance Verification

Results are saved in results/feature_importance_verification/dataset4/.

In [24]:
# Feature Importance Verification verification table and feature namespace audit.
D4_RESULTS_DIR=RESULTS_ROOT / "feature_importance_verification" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
d4_importance_audit = d4_perm_summary.join(
    d4_coef_summary, how="outer", lsuffix="_Permutation",
    rsuffix="_Coefficient"
).reset_index()
d4_importance_audit["Modality"] = np.where(
    d4_importance_audit["Feature"].isin(D4_ECOG), "ECOG", "Storyteller"
)
if d4_importance_audit["Feature"].duplicated().any():
    raise AssertionError("Duplicate Dataset 4 feature in importance audit.")
d4_importance_audit.to_csv(
    D4_RESULTS_DIR / "dataset4_feature_importance_verification_importance_verification.csv", index=False
)



C:\Users\amnar\AppData\Local\Temp\ipykernel_86684\2669748117.py:4: Pandas4Warning: The 'lsuffix' and 'rsuffix' keywords in DataFrame.join are deprecated and will be removed in a future version. Use the 'suffixes' keyword instead.
  d4_importance_audit = d4_perm_summary.join(


### Dataset 4: Feature Correlation and Redundancy

Results are saved in results/feature_correlation_redundancy/dataset4/.

In [25]:
# Feature Correlation and Redundancy Analysis correlations among stable-label participant-median profiles.
D4_RESULTS_DIR=RESULTS_ROOT / "feature_correlation_redundancy" / "dataset4"
D4_RESULTS_DIR.mkdir(parents=True,exist_ok=True)
def d4_bh(pvals):
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    adj = p[order] * len(p) / np.arange(1, len(p) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(len(p))
    out[order] = np.clip(adj, 0, 1)
    return out

corr_rows = []
for i, fa in enumerate(d4_part_X.columns):
    for fb in d4_part_X.columns[i + 1:]:
        pair = d4_part_X[[fa, fb]].dropna()
        if len(pair) < 5 or pair[fa].nunique() < 2 or pair[fb].nunique() < 2:
            continue
        pr, pp = pearsonr(pair[fa], pair[fb])
        sr, sp = spearmanr(pair[fa], pair[fb])
        ma = "ECOG" if fa in D4_ECOG else "Storyteller"
        mb = "ECOG" if fb in D4_ECOG else "Storyteller"
        corr_rows.append({
            "Feature_1": fa, "Feature_2": fb, "Analysis_Type": f"{ma}-{mb}",
            "N_Participants": len(pair), "Pearson_r": pr, "Pearson_p": pp,
            "Spearman_rho": sr, "Spearman_p": sp,
        })
d4_correlations = pd.DataFrame(corr_rows)
if not d4_correlations.empty:
    d4_correlations["Spearman_FDR"] = d4_bh(d4_correlations["Spearman_p"])
    d4_correlations["Absolute_Spearman"] = d4_correlations["Spearman_rho"].abs()
    d4_correlations = d4_correlations.sort_values("Absolute_Spearman", ascending=False)
    d4_high_correlations = d4_correlations[
        d4_correlations["Absolute_Spearman"] >= 0.70
    ].copy()
else:
    d4_high_correlations = d4_correlations.copy()
print("\nDATASET 4 EXPERIMENT 26")
print("Stable-label participants:", len(d4_part_X))
print("Valid feature pairs:", len(d4_correlations))
print("Pairs with |Spearman rho| >= 0.70:", len(d4_high_correlations))
display(d4_correlations.head(20).round(4))
display(d4_high_correlations[
    d4_high_correlations["Analysis_Type"] == "Storyteller-ECOG"
].round(4))
d4_correlations.to_csv(D4_RESULTS_DIR / "dataset4_feature_correlation_redundancy_all_correlations.csv", index=False)
d4_high_correlations.to_csv(
    D4_RESULTS_DIR / "dataset4_feature_correlation_redundancy_high_correlation_pairs.csv", index=False
)

print("\nDataset 4 Experiments 17-26 completed.")
print("Dataset 4 outputs:", D4_RESULTS_DIR)



DATASET 4 EXPERIMENT 26
Stable-label participants: 218
Valid feature pairs: 1953
Pairs with |Spearman rho| >= 0.70: 148


,Feature_1,Feature_2,Analysis_Type,N_Participants,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Spearman_FDR,Absolute_Spearman
1725,STORY1DEL_NUM_WORDS,STORY1DEL_SPEECH_DURATION,Storyteller-Storyteller,218,0.9508,0.0,0.9389,0.0,0.0,0.9389
1655,STORY1DEL_NUM_VERBS,STORY1DEL_NUM_WORDS,Storyteller-Storyteller,218,0.9399,0.0,0.9316,0.0,0.0,0.9316
1215,STORY2IMM_NUM_WORDS,STORY2IMM_SPEECH_DURATION,Storyteller-Storyteller,218,0.9253,0.0,0.9214,0.0,0.0,0.9214
416,STORY1IMM_NUM_WORDS,STORY1IMM_SPEECH_DURATION,Storyteller-Storyteller,218,0.9357,0.0,0.9213,0.0,0.0,0.9213
244,STORY1IMM_NUM_VERBS,STORY1IMM_NUM_WORDS,Storyteller-Storyteller,218,0.9122,0.0,0.9093,0.0,0.0,0.9093
127,STORY1IMM_NUM_NOUNS,STORY1IMM_NUM_WORDS,Storyteller-Storyteller,218,0.9080,0.0,0.9065,0.0,0.0,0.9065
1094,STORY2IMM_NUM_VERBS,STORY2IMM_NUM_WORDS,Storyteller-Storyteller,218,0.9030,0.0,0.8923,0.0,0.0,0.8923
1580,STORY1DEL_NUM_PRONOUNS,STORY1DEL_NUM_WORDS,Storyteller-Storyteller,218,0.9044,0.0,0.8912,0.0,0.0,0.8912
67,STORY1IMM_NUM_PRONOUNS,STORY1IMM_NUM_WORDS,Storyteller-Storyteller,218,0.8996,0.0,0.8881,0.0,0.0,0.8881
1011,STORY2IMM_NUM_NOUNS,STORY2IMM_NUM_WORDS,Storyteller-Storyteller,218,0.9048,0.0,0.8847,0.0,0.0,0.8847


,Feature_1,Feature_2,Analysis_Type,N_Participants,Pearson_r,Pearson_p,Spearman_rho,Spearman_p,Spearman_FDR,Absolute_Spearman



Dataset 4 Experiments 17-26 completed.
Dataset 4 outputs: c:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\results\feature_correlation_redundancy\dataset4
